# 06 GenAI Inventory RAG Documents

This notebook creates an inventory RAG document table from Gold inventory KPI tables.

The goal is to convert structured inventory records into readable text documents that can be retrieved later by a GenAI assistant.

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retail_capstone")

spark.sql("SELECT current_catalog(), current_schema()").show()

+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace| retail_capstone|
+-----------------+----------------+



In [0]:
from pyspark.sql.functions import col, concat, lit, coalesce, round, when

In [0]:
inventory_status_df = spark.table("workspace.retail_capstone.gold_inventory_status")
sales_velocity_df = spark.table("workspace.retail_capstone.gold_product_sales_velocity")
stockout_risk_df = spark.table("workspace.retail_capstone.gold_stockout_risk")
reorder_df = spark.table("workspace.retail_capstone.gold_reorder_recommendations")
inventory_value_df = spark.table("workspace.retail_capstone.gold_inventory_value")

In [0]:
display(stockout_risk_df)

stock_code,product_name,category,brand,warehouse_id,warehouse_name,available_stock,avg_daily_sales,days_of_inventory_remaining,supplier_name,lead_time_days,reliability_score,stockout_risk_level
84029G,KNITTED UNION FLAG HOT WATER BOTTLE,Home & Living,WarmHome,WH002,Berlin Distribution Hub,25,17.91,1.4,WarmHome Manufacturing,14,0.91,High Risk
85123A,WHITE HANGING HEART T-LIGHT HOLDER,Home Decor,Generic Home,WH001,Nuremberg Fulfillment Center,210,120.15,1.75,Global Home Supplies,7,0.94,High Risk
84029E,RED WOOLLY HOTTIE WHITE HEART,Home & Living,WarmHome,WH002,Berlin Distribution Hub,48,38.24,1.26,WarmHome Manufacturing,14,0.91,High Risk
84879,ASSORTED COLOUR BIRD ORNAMENT,Home Decor,Generic Home,WH001,Nuremberg Fulfillment Center,100,117.5,0.85,Global Home Supplies,7,0.94,High Risk
71053,WHITE METAL LANTERN,Home Decor,Generic Home,WH001,Nuremberg Fulfillment Center,75,10.41,7.2,Global Home Supplies,7,0.94,Medium Risk
21730,GLASS STAR FROSTED T-LIGHT HOLDER,Home Decor,GlassWorks,WH001,Nuremberg Fulfillment Center,110,6.29,17.49,GlassWorks Studio,9,0.89,Low Risk
22752,SET 7 BABUSHKA NESTING BOXES,Gifts,GiftCraft,WH003,Cologne Regional Warehouse,20,9.2,2.17,GiftCraft Wholesale,12,0.86,High Risk
84406B,CREAM CUPID HEARTS COAT HANGER,Home Decor,Generic Home,WH002,Berlin Distribution Hub,40,12.52,3.19,DecorCraft Europe,10,0.88,High Risk
22633,HAND WARMER UNION JACK,Accessories,WarmHome,WH003,Cologne Regional Warehouse,60,45.52,1.32,WarmHome Manufacturing,14,0.91,High Risk
22632,HAND WARMER RED POLKA DOT,Accessories,WarmHome,WH003,Cologne Regional Warehouse,50,43.69,1.14,WarmHome Manufacturing,14,0.91,High Risk


In [0]:
rag_base_df = stockout_risk_df.join(
    reorder_df.select(
        "stock_code",
        "warehouse_id",
        "reorder_level",
        "reorder_quantity",
        "recommended_reorder_quantity",
        "reorder_flag"
    ),
    on=["stock_code", "warehouse_id"],
    how="left"
).join(
    inventory_value_df.select(
        "stock_code",
        "warehouse_id",
        "current_stock",
        "unit_cost",
        "inventory_value"
    ),
    on=["stock_code", "warehouse_id"],
    how="left"
)

In [0]:
display(rag_base_df)

stock_code,warehouse_id,product_name,category,brand,warehouse_name,available_stock,avg_daily_sales,days_of_inventory_remaining,supplier_name,lead_time_days,reliability_score,stockout_risk_level,reorder_level,reorder_quantity,recommended_reorder_quantity,reorder_flag,current_stock,unit_cost,inventory_value
84029G,WH002,KNITTED UNION FLAG HOT WATER BOTTLE,Home & Living,WarmHome,Berlin Distribution Hub,25,17.91,1.4,WarmHome Manufacturing,14,0.91,High Risk,50,120,120,Reorder Needed,35,3.5,122.5
85123A,WH001,WHITE HANGING HEART T-LIGHT HOLDER,Home Decor,Generic Home,Nuremberg Fulfillment Center,210,120.15,1.75,Global Home Supplies,7,0.94,High Risk,100,300,0,No Reorder Needed,250,1.25,312.5
84029E,WH002,RED WOOLLY HOTTIE WHITE HEART,Home & Living,WarmHome,Berlin Distribution Hub,48,38.24,1.26,WarmHome Manufacturing,14,0.91,High Risk,50,120,120,Reorder Needed,60,3.25,195.0
84879,WH001,ASSORTED COLOUR BIRD ORNAMENT,Home Decor,Generic Home,Nuremberg Fulfillment Center,100,117.5,0.85,Global Home Supplies,7,0.94,High Risk,90,250,0,No Reorder Needed,110,1.5,165.0
71053,WH001,WHITE METAL LANTERN,Home Decor,Generic Home,Nuremberg Fulfillment Center,75,10.41,7.2,Global Home Supplies,7,0.94,Medium Risk,80,200,200,Reorder Needed,90,2.1,189.0
21730,WH001,GLASS STAR FROSTED T-LIGHT HOLDER,Home Decor,GlassWorks,Nuremberg Fulfillment Center,110,6.29,17.49,GlassWorks Studio,9,0.89,Low Risk,70,180,0,No Reorder Needed,140,2.75,385.0
22752,WH003,SET 7 BABUSHKA NESTING BOXES,Gifts,GiftCraft,Cologne Regional Warehouse,20,9.2,2.17,GiftCraft Wholesale,12,0.86,High Risk,40,100,100,Reorder Needed,25,4.0,100.0
84406B,WH002,CREAM CUPID HEARTS COAT HANGER,Home Decor,Generic Home,Berlin Distribution Hub,40,12.52,3.19,DecorCraft Europe,10,0.88,High Risk,60,150,150,Reorder Needed,45,1.85,83.25
22633,WH003,HAND WARMER UNION JACK,Accessories,WarmHome,Cologne Regional Warehouse,60,45.52,1.32,WarmHome Manufacturing,14,0.91,High Risk,120,300,300,Reorder Needed,80,1.1,88.0
22632,WH003,HAND WARMER RED POLKA DOT,Accessories,WarmHome,Cologne Regional Warehouse,50,43.69,1.14,WarmHome Manufacturing,14,0.91,High Risk,120,300,300,Reorder Needed,75,1.1,82.5


In [0]:
rag_scored_df = rag_base_df.withColumn(
    "risk_score",
    when(col("stockout_risk_level") == "High Risk", 1.0)
    .when(col("stockout_risk_level") == "Medium Risk", 0.75)
    .when(col("stockout_risk_level") == "Low Risk", 0.40)
    .when(col("stockout_risk_level") == "No Recent Sales", 0.20)
    .otherwise(0.10)
).withColumn(
    "reorder_score",
    when(col("reorder_flag") == "Reorder Needed", 1.0)
    .otherwise(0.0)
).withColumn(
    "supplier_risk_score",
    1 - col("reliability_score")
).withColumn(
    "business_priority_score",
    round(
        (col("risk_score") * 0.6) +
        (col("reorder_score") * 0.3) +
        (col("supplier_risk_score") * 0.1),
        3
    )
)

In [0]:
display(
    rag_scored_df.select(
        "stock_code",
        "product_name",
        "warehouse_id",
        "stockout_risk_level",
        "reorder_flag",
        "reliability_score",
        "business_priority_score"
    )
)

stock_code,product_name,warehouse_id,stockout_risk_level,reorder_flag,reliability_score,business_priority_score
84029G,KNITTED UNION FLAG HOT WATER BOTTLE,WH002,High Risk,Reorder Needed,0.91,0.909
85123A,WHITE HANGING HEART T-LIGHT HOLDER,WH001,High Risk,No Reorder Needed,0.94,0.606
84029E,RED WOOLLY HOTTIE WHITE HEART,WH002,High Risk,Reorder Needed,0.91,0.909
84879,ASSORTED COLOUR BIRD ORNAMENT,WH001,High Risk,No Reorder Needed,0.94,0.606
71053,WHITE METAL LANTERN,WH001,Medium Risk,Reorder Needed,0.94,0.756
21730,GLASS STAR FROSTED T-LIGHT HOLDER,WH001,Low Risk,No Reorder Needed,0.89,0.251
22752,SET 7 BABUSHKA NESTING BOXES,WH003,High Risk,Reorder Needed,0.86,0.914
84406B,CREAM CUPID HEARTS COAT HANGER,WH002,High Risk,Reorder Needed,0.88,0.912
22633,HAND WARMER UNION JACK,WH003,High Risk,Reorder Needed,0.91,0.909
22632,HAND WARMER RED POLKA DOT,WH003,High Risk,Reorder Needed,0.91,0.909


In [0]:
rag_documents_df = rag_scored_df.withColumn(
    "document_id",
    concat(
        col("stock_code"),
        lit("_"),
        col("warehouse_id")
    )
).withColumn(
    "rag_document_text",
    concat(
        lit("Product "),
        col("stock_code"),
        lit(", "),
        col("product_name"),
        lit(", belongs to category "),
        col("category"),
        lit(" and brand "),
        col("brand"),
        lit(". It is stored in warehouse "),
        col("warehouse_id"),
        lit(" named "),
        col("warehouse_name"),
        lit(". Available stock is "),
        col("available_stock").cast("string"),
        lit(" units. Current stock is "),
        col("current_stock").cast("string"),
        lit(" units. Average daily sales is "),
        col("avg_daily_sales").cast("string"),
        lit(" units. Days of inventory remaining is "),
        col("days_of_inventory_remaining").cast("string"),
        lit(". Supplier is "),
        col("supplier_name"),
        lit(" with lead time of "),
        col("lead_time_days").cast("string"),
        lit(" days and reliability score of "),
        col("reliability_score").cast("string"),
        lit(". Stockout risk level is "),
        col("stockout_risk_level"),
        lit(". Reorder status is "),
        col("reorder_flag"),
        lit(". Recommended reorder quantity is "),
        col("recommended_reorder_quantity").cast("string"),
        lit(" units. Inventory value is "),
        col("inventory_value").cast("string"),
        lit(".")
    )
)

In [0]:
inventory_rag_documents_df = rag_documents_df.select(
    "document_id",
    "stock_code",
    "product_name",
    "category",
    "brand",
    "warehouse_id",
    "warehouse_name",
    "available_stock",
    "avg_daily_sales",
    "days_of_inventory_remaining",
    "supplier_name",
    "lead_time_days",
    "reliability_score",
    "stockout_risk_level",
    "reorder_flag",
    "recommended_reorder_quantity",
    "inventory_value",
    "business_priority_score",
    "rag_document_text"
)

In [0]:
display(inventory_rag_documents_df)

document_id,stock_code,product_name,category,brand,warehouse_id,warehouse_name,available_stock,avg_daily_sales,days_of_inventory_remaining,supplier_name,lead_time_days,reliability_score,stockout_risk_level,reorder_flag,recommended_reorder_quantity,inventory_value,business_priority_score,rag_document_text
84029G_WH002,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,Home & Living,WarmHome,WH002,Berlin Distribution Hub,25,17.91,1.4,WarmHome Manufacturing,14,0.91,High Risk,Reorder Needed,120,122.5,0.909,"Product 84029G, KNITTED UNION FLAG HOT WATER BOTTLE, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 25 units. Current stock is 35 units. Average daily sales is 17.91 units. Days of inventory remaining is 1.4. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 122.5."
85123A_WH001,85123A,WHITE HANGING HEART T-LIGHT HOLDER,Home Decor,Generic Home,WH001,Nuremberg Fulfillment Center,210,120.15,1.75,Global Home Supplies,7,0.94,High Risk,No Reorder Needed,0,312.5,0.606,"Product 85123A, WHITE HANGING HEART T-LIGHT HOLDER, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 210 units. Current stock is 250 units. Average daily sales is 120.15 units. Days of inventory remaining is 1.75. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is High Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 312.5."
84029E_WH002,84029E,RED WOOLLY HOTTIE WHITE HEART,Home & Living,WarmHome,WH002,Berlin Distribution Hub,48,38.24,1.26,WarmHome Manufacturing,14,0.91,High Risk,Reorder Needed,120,195.0,0.909,"Product 84029E, RED WOOLLY HOTTIE WHITE HEART, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 48 units. Current stock is 60 units. Average daily sales is 38.24 units. Days of inventory remaining is 1.26. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 195.0."
84879_WH001,84879,ASSORTED COLOUR BIRD ORNAMENT,Home Decor,Generic Home,WH001,Nuremberg Fulfillment Center,100,117.5,0.85,Global Home Supplies,7,0.94,High Risk,No Reorder Needed,0,165.0,0.606,"Product 84879, ASSORTED COLOUR BIRD ORNAMENT, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 100 units. Current stock is 110 units. Average daily sales is 117.5 units. Days of inventory remaining is 0.85. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is High Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 165.0."
71053_WH001,71053,WHITE METAL LANTERN,Home Decor,Generic Home,WH001,Nuremberg Fulfillment Center,75,10.41,7.2,Global Home Supplies,7,0.94,Medium Risk,Reorder Needed,200,189.0,0.756,"Product 71053, WHITE METAL LANTERN, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 75 units. Current stock is 90 units. Average daily sales is 10.41 units. Days of inventory remaining is 7.2. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is Medium Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 200 units. Inventory value is 189.0."
21730_WH001,21730,GLASS STAR FROSTED T-LIGHT HOLDER,Home Decor,GlassWorks,WH001,Nuremberg Fulfillment C

In [0]:
inventory_rag_documents_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail_capstone.inventory_rag_documents")

In [0]:
spark.sql("""
SELECT COUNT(*) AS document_count
FROM workspace.retail_capstone.inventory_rag_documents
""").display()

document_count
10


In [0]:
display(
    spark.table("workspace.retail_capstone.inventory_rag_documents")
    .select("document_id", "business_priority_score", "rag_document_text")
)

document_id,business_priority_score,rag_document_text
84029G_WH002,0.909,"Product 84029G, KNITTED UNION FLAG HOT WATER BOTTLE, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 25 units. Current stock is 35 units. Average daily sales is 17.91 units. Days of inventory remaining is 1.4. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 122.5."
85123A_WH001,0.606,"Product 85123A, WHITE HANGING HEART T-LIGHT HOLDER, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 210 units. Current stock is 250 units. Average daily sales is 120.15 units. Days of inventory remaining is 1.75. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is High Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 312.5."
84029E_WH002,0.909,"Product 84029E, RED WOOLLY HOTTIE WHITE HEART, belongs to category Home & Living and brand WarmHome. It is stored in warehouse WH002 named Berlin Distribution Hub. Available stock is 48 units. Current stock is 60 units. Average daily sales is 38.24 units. Days of inventory remaining is 1.26. Supplier is WarmHome Manufacturing with lead time of 14 days and reliability score of 0.91. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 120 units. Inventory value is 195.0."
84879_WH001,0.606,"Product 84879, ASSORTED COLOUR BIRD ORNAMENT, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 100 units. Current stock is 110 units. Average daily sales is 117.5 units. Days of inventory remaining is 0.85. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is High Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 165.0."
71053_WH001,0.756,"Product 71053, WHITE METAL LANTERN, belongs to category Home Decor and brand Generic Home. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 75 units. Current stock is 90 units. Average daily sales is 10.41 units. Days of inventory remaining is 7.2. Supplier is Global Home Supplies with lead time of 7 days and reliability score of 0.94. Stockout risk level is Medium Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 200 units. Inventory value is 189.0."
21730_WH001,0.251,"Product 21730, GLASS STAR FROSTED T-LIGHT HOLDER, belongs to category Home Decor and brand GlassWorks. It is stored in warehouse WH001 named Nuremberg Fulfillment Center. Available stock is 110 units. Current stock is 140 units. Average daily sales is 6.29 units. Days of inventory remaining is 17.49. Supplier is GlassWorks Studio with lead time of 9 days and reliability score of 0.89. Stockout risk level is Low Risk. Reorder status is No Reorder Needed. Recommended reorder quantity is 0 units. Inventory value is 385.0."
22752_WH003,0.914,"Product 22752, SET 7 BABUSHKA NESTING BOXES, belongs to category Gifts and brand GiftCraft. It is stored in warehouse WH003 named Cologne Regional Warehouse. Available stock is 20 units. Current stock is 25 units. Average daily sales is 9.2 units. Days of inventory remaining is 2.17. Supplier is GiftCraft Wholesale with lead time of 12 days and reliability score of 0.86. Stockout risk level is High Risk. Reorder status is Reorder Needed. Recommended reorder quantity is 100 units. Inventory value is 100.0."
84406B_WH002,0.912,"Product 84406B, CREAM CUPID HEARTS COAT HANGER, belongs to category Home Decor and brand Generic Home. It is stored in war

# Inventory RAG Document Table Completed

This notebook created the `inventory_rag_documents` Delta table.

Each row is a natural language document created from Gold inventory KPI tables.

The table includes:

- Product details
- Warehouse details
- Available stock
- Sales velocity
- Days of inventory remaining
- Supplier lead time
- Stockout risk level
- Reorder flag
- Recommended reorder quantity
- Inventory value
- Business priority score

The business priority score is used later to rank high-risk inventory records more strongly during retrieval.